In [11]:
# Mount Google Drive to save your model
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:
import torch
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

GPU Available: True
GPU Name: Tesla T4


Cell 2: Imports

In [37]:
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
from sklearn.metrics import accuracy_score, f1_score, classification_report
import gradio as gr
import time

Cell 3: Load & Explore Dataset

In [28]:
# Load AG News dataset
print("Loading AG News dataset...")
dataset = load_dataset("ag_news")

# Label mapping
id2label = {0: "World", 1: "Sports", 2: "Business", 3: "Sci/Tech"}
label2id = {v: k for k, v in id2label.items()}

print(f"\n✅ Dataset loaded successfully!")
print(f"Training samples: {len(dataset['train']):,}")
print(f"Test samples: {len(dataset['test']):,}")
print(f"\nCategories: {id2label}")

# Show a sample
print("\n📰 Sample headline:")
print(f"Text: {dataset['train'][0]['text']}")
print(f"Label: {id2label[dataset['train'][0]['label']]}")

Loading AG News dataset...

✅ Dataset loaded successfully!
Training samples: 120,000
Test samples: 7,600

Categories: {0: 'World', 1: 'Sports', 2: 'Business', 3: 'Sci/Tech'}

📰 Sample headline:
Text: Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\band of ultra-cynics, are seeing green again.
Label: Business


Cell 4: Tokenization

In [29]:
# Use DistilBERT
model_checkpoint = "distilbert-base-uncased"
print(f"Using model: {model_checkpoint}")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding=True,
        max_length=256  # Can reduce to 128 for even faster training
    )

print("Tokenizing dataset...")
start_time = time.time()
tokenized_datasets = dataset.map(tokenize_function, batched=True)
print(f"✅ Tokenization completed in {time.time() - start_time:.2f} seconds")

# Prepare datasets
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

train_dataset = tokenized_datasets["train"]
eval_dataset = tokenized_datasets["test"].select(range(1000))  # Use 1000 samples for validation

print(f"\nTraining samples: {len(train_dataset):,}")
print(f"Validation samples: {len(eval_dataset):,}")

Using model: distilbert-base-uncased


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizing dataset...


Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

✅ Tokenization completed in 91.08 seconds

Training samples: 120,000
Validation samples: 1,000


Cell 5: Load Pre-trained Model ,Cell 5: Load Model

In [30]:
# Load DistilBERT model for classification
print("Loading DistilBERT model...")
model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint,
    num_labels=4,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\n✅ Model loaded successfully!")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Model size: {total_params * 4 / (1024**2):.1f} MB")

Loading DistilBERT model...


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



✅ Model loaded successfully!
Total parameters: 66,956,548
Trainable parameters: 66,956,548
Model size: 255.4 MB


Cell 6: Training Configuration

Cell 6: Define Metrics & Training Arguments

In [31]:
# Define evaluation metrics
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average="weighted")
    return {
        "accuracy": acc,
        "f1_score": f1
    }

# Optimized training arguments for speed
training_args = TrainingArguments(
    output_dir="./ag_news_model",
    learning_rate=2e-5,
    per_device_train_batch_size=32,  # Increased for faster training
    per_device_eval_batch_size=128,
    num_train_epochs=2,  # Only 2 epochs (enough for good accuracy)
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    logging_steps=50,
    report_to="none",  # Disable external logging
    fp16=True,  # Mixed precision training
    dataloader_num_workers=2,
    warmup_ratio=0.1,
)

# Data collator for dynamic padding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print("✅ Training configuration ready!")
print(f"Batch size: {training_args.per_device_train_batch_size}")
print(f"Epochs: {training_args.num_train_epochs}")
print(f"Learning rate: {training_args.learning_rate}")
print(f"Total training steps: ~{len(train_dataset) // training_args.per_device_train_batch_size * training_args.num_train_epochs:,}")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


✅ Training configuration ready!
Batch size: 32
Epochs: 2
Learning rate: 2e-05
Total training steps: ~7,500


Cell 7: Train

In [32]:
# Initialize trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("\n🚀 Starting training...")
print("="*50)
print(f"Expected training time: ~20-25 minutes")
print("="*50)

# Start training
start_time = time.time()
trainer.train()
training_time = time.time() - start_time

print(f"\n✅ Training completed in {training_time / 60:.1f} minutes!")




🚀 Starting training...
Expected training time: ~20-25 minutes


Epoch,Training Loss,Validation Loss,Accuracy,F1 Score
1,0.196167,0.186100,0.936000,0.935869
2,0.128605,0.178441,0.945000,0.944967


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].



✅ Training completed in 23.5 minutes!


In [33]:
# Save model locally
print("\n💾 Saving model...")
model.save_pretrained("./my_agnews_model")
tokenizer.save_pretrained("./my_agnews_model")
print("✅ Model saved to ./my_agnews_model")

# Optional: Save to Google Drive
try:
    model.save_pretrained("/content/drive/MyDrive/ag_news_model")
    tokenizer.save_pretrained("/content/drive/MyDrive/ag_news_model")
    print("✅ Model also saved to Google Drive")
except:
    print("⚠️ Could not save to Google Drive (make sure it's mounted)")


💾 Saving model...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model saved to ./my_agnews_model


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model also saved to Google Drive


Cell 8: Evaluate Model

In [34]:
print("\n📊 Evaluating on test set...")
test_results = trainer.evaluate(tokenized_datasets["test"])

print("\n" + "="*50)
print("FINAL RESULTS")
print("="*50)
print(f"Test Accuracy:  {test_results['eval_accuracy']:.4f} ({test_results['eval_accuracy']*100:.2f}%)")
print(f"Test F1-Score:  {test_results['eval_f1_score']:.4f}")
print(f"Test Loss:      {test_results['eval_loss']:.4f}")

# Get detailed classification report
print("\n📈 Detailed Classification Report:")
predictions = trainer.predict(tokenized_datasets["test"])
pred_labels = np.argmax(predictions.predictions, axis=1)
true_labels = predictions.label_ids

print(classification_report(
    true_labels,
    pred_labels,
    target_names=[id2label[i] for i in range(4)]
))


📊 Evaluating on test set...



FINAL RESULTS
Test Accuracy:  0.9462 (94.62%)
Test F1-Score:  0.9462
Test Loss:      0.1708

📈 Detailed Classification Report:
              precision    recall  f1-score   support

       World       0.96      0.95      0.96      1900
      Sports       0.99      0.99      0.99      1900
    Business       0.93      0.91      0.92      1900
    Sci/Tech       0.91      0.93      0.92      1900

    accuracy                           0.95      7600
   macro avg       0.95      0.95      0.95      7600
weighted avg       0.95      0.95      0.95      7600



Cell 9: Quick Testing

In [36]:
# Cell 9: Test Your Model
from transformers import pipeline

# Load the saved model
classifier = pipeline(
    "text-classification",
    model="./my_agnews_model",
    tokenizer="./my_agnews_model"
)

# Test headlines
test_headlines = [
    "Apple announces new iPhone with advanced AI features",
    "Manchester United wins Premier League title in dramatic finale",
    "US and China agree to new trade deal negotiations",
    "Global stock markets rally after Federal Reserve announcement",
    "Scientists discover breakthrough in quantum computing",
    "World Cup 2026 to be hosted by three countries"
]

print("\n🧪 Testing model with custom headlines:")
print("="*60)

for headline in test_headlines:
    result = classifier(headline)[0]

    # FIXED: Your model outputs label names directly (e.g., "Sci/Tech", "Sports")
    # So we don't need to convert anything!
    predicted_category = result['label']  # ← Directly use the label
    confidence = result['score']

    print(f"\n📰 Headline: {headline}")
    print(f"✅ Predicted: {predicted_category} (confidence: {confidence:.2%})")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]


🧪 Testing model with custom headlines:

📰 Headline: Apple announces new iPhone with advanced AI features
✅ Predicted: Sci/Tech (confidence: 96.74%)

📰 Headline: Manchester United wins Premier League title in dramatic finale
✅ Predicted: World (confidence: 86.22%)

📰 Headline: US and China agree to new trade deal negotiations
✅ Predicted: Business (confidence: 84.80%)

📰 Headline: Global stock markets rally after Federal Reserve announcement
✅ Predicted: Business (confidence: 99.48%)

📰 Headline: Scientists discover breakthrough in quantum computing
✅ Predicted: Sci/Tech (confidence: 93.97%)

📰 Headline: World Cup 2026 to be hosted by three countries
✅ Predicted: World (confidence: 93.07%)


Cell 10: Gradio Web App

In [38]:
# Cell 10: Gradio Web App
from transformers import pipeline
import gradio as gr

# Load your trained model
classifier = pipeline("text-classification", model="./my_agnews_model")

# Your model already outputs labels like "World", "Sports", "Business", "Sci/Tech"
# But let's add emojis for better display
label_emojis = {
    "World": "🌍 World",
    "Sports": "⚽ Sports",
    "Business": "💼 Business",
    "Sci/Tech": "🔬 Sci/Tech"
}

def predict_news_category(news_headline):
    result = classifier(news_headline)[0]
    # result['label'] is already "World", "Sports", "Business", or "Sci/Tech"
    category = result['label']
    confidence = result['score']

    # Add emoji for better display
    display_name = label_emojis.get(category, category)

    return {display_name: confidence}

# Create the interface
iface = gr.Interface(
    fn=predict_news_category,
    inputs=gr.Textbox(
        lines=3,
        placeholder="Enter a news headline here...",
        label="📰 News Headline"
    ),
    outputs=gr.Label(num_top_classes=4, label="🎯 Predicted Category"),
    title="📰 News Topic Classifier - Your Fine-tuned Model",
    description=f"""
    ### 🎉 Your Model Achieved 94.6% Accuracy!

    **Test Results:**
    - 🌍 World News: 96% precision
    - ⚽ Sports: 99% precision
    - 💼 Business: 93% precision
    - 🔬 Sci/Tech: 91% precision

    Just enter any news headline below!
    """,
    examples=[
        ["Tesla stock surges after record quarterly earnings"],
        ["Manchester United wins Premier League title"],
        ["NASA discovers new exoplanet in habitable zone"],
        ["US and China agree to new trade deal"],
        ["Scientists find breakthrough in cancer research"],
        ["Olympic Games postponed due to weather"],
        ["Bitcoin price reaches all-time high"],
        ["New AI model can generate realistic images"]
    ],
    allow_flagging="never"
)

print("\n" + "="*60)
print("🚀 SUCCESS! Your Gradio app is ready!")
print("="*60)
print("📱 Share this link with anyone:")
print("   They can use your model from any device (phone, tablet, laptop)")
print("   No installation needed!")
print("="*60)

# Launch the app
iface.launch(share=True)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/gradio/interface.py:415: UserWarning: The `allow_flagging` parameter in `Interface` is deprecated. Use `flagging_mode` instead.
  warnings.warn(



🚀 SUCCESS! Your Gradio app is ready!
📱 Share this link with anyone:
   They can use your model from any device (phone, tablet, laptop)
   No installation needed!
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://afe0c5e7583bc169b3.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
